In [1]:
import pandas as pd

OVERLAP_FILE = "dialog_level_overlap_no_turn.csv"
GRICE_FILE = "grice_dialog_level.csv"
OUTPUT = "merged_dialog_analysis.csv"

overlap = pd.read_csv(OVERLAP_FILE)
grice = pd.read_csv(GRICE_FILE)

grice = grice.rename(columns={"_dialog_id": "dialog_id"})

merged = overlap.merge(grice, on="dialog_id", how="inner")

print("Merged shape:", merged.shape)
print("\nRows per mode:")
print(merged["mode"].value_counts())

print("\nUnique dialog IDs:", merged["dialog_id"].nunique())

print("\nColumns:")
print(merged.columns.tolist())

print("\nFirst 10 rows:")
print(merged.head(10))

merged.to_csv(OUTPUT, index=False, encoding="utf-8")
print("\nSaved file:", OUTPUT)

Merged shape: (31314, 12)

Rows per mode:
mode
strict        10438
normalized    10438
lenient       10438
Name: count, dtype: int64

Unique dialog IDs: 10438

Columns:
['dialog_id', 'mode', 'ref_len', 'hyp_len', 'jaccard', 'bleu_dialog', 'grice_dialog_mean', 'grice_dialog_median', 'grice_dialog_min', 'grice_dialog_max', 'grice_turn_count', 'grice_total_units']

First 10 rows:
      dialog_id    mode  ref_len  hyp_len   jaccard  bleu_dialog  \
0  MUL0001.json  strict       68       59  0.352941    22.116362   
1  MUL0002.json  strict       48       38  0.270270    14.266022   
2  MUL0003.json  strict       74       50  0.428571    20.273414   
3  MUL0004.json  strict       63       42  0.340000    14.898419   
4  MUL0005.json  strict       74       58  0.285714    20.457801   
5  MUL0006.json  strict       69       51  0.327273    16.054153   
6  MUL0007.json  strict       51       40  0.333333    18.365598   
7  MUL0008.json  strict       98       69  0.338462    14.311300   
8  MUL00

In [2]:
import pandas as pd
from scipy.stats import spearmanr

FILE = "merged_dialog_analysis.csv"

df = pd.read_csv(FILE)

rows = []

for mode in ["strict", "normalized", "lenient"]:
    sub = df[df["mode"] == mode].copy()

    rho_j, p_j = spearmanr(sub["jaccard"], sub["grice_dialog_mean"])
    rho_b, p_b = spearmanr(sub["bleu_dialog"], sub["grice_dialog_mean"])

    rows.append({
        "mode": mode,
        "n": len(sub),
        "rho_jaccard_grice": rho_j,
        "p_jaccard_grice": p_j,
        "rho_bleu_grice": rho_b,
        "p_bleu_grice": p_b,
    })

corr_results = pd.DataFrame(rows)

print(corr_results)

corr_results.to_csv("spearman_results.csv", index=False, encoding="utf-8")
print("\nSaved: spearman_results.csv")

         mode      n  rho_jaccard_grice  p_jaccard_grice  rho_bleu_grice  \
0      strict  10438          -0.097514     1.768257e-23       -0.009758   
1  normalized  10438          -0.079174     5.463722e-16       -0.022918   
2     lenient  10438          -0.065183     2.632630e-11       -0.013395   

   p_bleu_grice  
0      0.318851  
1      0.019209  
2      0.171166  

Saved: spearman_results.csv
